# W04 · 02 주기경계 · 컷오프 · 꼬리 보정
목표 3
1. W03 과 같은 탄소 LJ 유체(ε 0.109 · σ 3.40)를 216입자 · L 26.2 Å 상자에 만들고 최소 이미지 규약으로 거리 계산
2. 컷오프 6 · 8 · 10 · 12 Å 로 퍼텐셜 합 → 절단 손실 (슬라이드 23: r⁻⁶ 은 12 Å 에서 거의 0)
3. 꼬리 보정 U_tail = (8/3)πNρεσ³[(1/3)(σ/r_c)⁹ − (σ/r_c)³] 을 더하면 차이 < 1 % → `out/cutoff_scan.json`

규칙 ③(슬라이드 22): 컷오프 ≤ L/2. W03 상자(64입자 · L 17.5 Å)는 L/2 = 8.7 Å 라 10 · 12 Å 를 쓸 수 없다.

실행(기본): week04 폴더에서 연 Claude Code 에 「`@notebooks/02_pbc_cutoff.ipynb` 를 실행하고 결과를 요약해」 → 승인 → `python ../run_nb.py notebooks/02_pbc_cutoff.ipynb` · 그림을 직접 보려면 `jupyter lab`

In [ ]:
# 기본 = 내 컴퓨터 mdai 환경 · Claude Code 가 week04 폴더에서 실행 · 아래 google.colab 분기는 Colab(선택)에서만 동작
import os, sys, json
if "google.colab" in sys.modules and not os.path.exists("scripts/ff_tools.py"):
    !git clone https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git repo && %cd repo/week04
import numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "scripts")
from ff_tools import *   # load_types mix mixing_table cutoff_scan potential tail_correction OUT

In [ ]:
res = cutoff_scan()                 # 약 30초
print("N", res["N"], "L", res["L_A"], "A · L/2 =", res["min_image_limit_A"], "A · 기준 U", res["U_ref_rc_L2_plus_tail"])
for r in res["rows"]:
    print(f'rc {r["rc_A"]:5.1f}  쌍 {r["pairs_in_cutoff"]:5d}  U_cut {r["U_cut"]:9.3f}  tail {r["U_tail"]:8.3f}  절단만 {r["rel_diff_vs_ref_pct"]:7.2f} %  꼬리 보정 후 {r["rel_diff_with_tail_pct"]:7.3f} %')

In [ ]:
rc = [r["rc_A"] for r in res["rows"]]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(rc, [r["rel_diff_vs_ref_pct"] for r in res["rows"]], "o-", label="cutoff only")
ax.plot(rc, [r["rel_diff_with_tail_pct"] for r in res["rows"]], "s-", label="cutoff + tail correction")
ax.axhline(0, color="gray", lw=0.8); ax.set_xlabel("cutoff (A)"); ax.set_ylabel("U error vs reference (%)"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "cutoff_scan.png", dpi=120); plt.show()

## 읽기
- 절단만: 6 Å 에서 30 % · 12 Å 에서 4 % 손실 — r⁻⁶ 라도 쌍 수가 r² 로 늘어 껍질 기여는 r⁻⁴ 로만 줄어든다
- 꼬리 보정 후: 8 Å 에서 0.1 % · 12 Å 에서 0.003 % — 균일 밀도 가정이 유효한 유체에서는 절단 + 보정으로 충분
- 쿨롱 r⁻¹ 은 껍질 기여가 r 로 커져 합이 수렴하지 않는다 → 보정이 아니라 분할(PPPM · 슬라이드 24)

In [ ]:
# 선택: W03 상자(64입자 · L 17.5 Å)에서 같은 스캔 — rc 10 · 12 Å 는 L/2 를 넘어 유효하지 않다
r64 = cutoff_scan(n_side=4); print("L/2 =", r64["min_image_limit_A"], [ (r["rc_A"], r["valid_rc_le_L2"]) for r in r64["rows"] ])